[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/) [![Python 3.10](https://img.shields.io/badge/python-3.10-blue.svg)](https://python.org) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)

# A Predictive Supply Chain Framework: Leveraging Small Language Models for Sentiment-Based Demand Forecasting

**Authors:** Bhavana Tiple, Riya Kulkarni, Naitri Panchal, Heramb Churi  
**Institution:** MIT World Peace University, Pune, India  
**Contact:** bhavana.tiple@mitwpu.edu.in

---

## Overview

This notebook is the complete, reproducible implementation of the methodology described in our research paper. It addresses the **Intermittency Problem** in retail demand forecasting — where high-frequency noise and sparse data limit purely numerical models.

**Core Idea:** Combine a Small Language Model (Microsoft Phi-3-mini, 4-bit quantized) with historical time-series sales data in a multimodal late-fusion architecture to improve weekly demand forecasting.

**Research Question:** Can semantic representations of product metadata derived by an SLM outperform demand-forecasting baselines?

**Experimental Design:**
- Two product categories: **All Beauty** (hedonic goods) vs **Electronics** (utilitarian goods)
- Amazon Product Research Dataset 2023
- Evaluation metric: Mean Absolute Error (MAE)

### Key Results

| Category | Baseline MAE | Multimodal MAE | Change |
|---|---|---|---|
| All Beauty (hedonic) | 4.59 | 4.46 | **−2.83% ✅** |
| Electronics (utilitarian) | 3.20 | 3.31 | +3.44% ❌ |

---

## Notebook Structure
1. [Configuration](#config) — paths, hyperparameters, seeds
2. [Environment Setup](#env) — dependencies, GPU check
3. [Data Acquisition & Preprocessing](#data) — Paper §II-B, II-C
4. [Temporal Alignment](#temporal) — daily → weekly, Paper §II-D
5. [SLM Semantic Inference](#slm) — Phi-3-mini 4-bit NF4, Paper §II-E
6. [Multimodal Feature Fusion](#fusion) — Paper §II-F, II-G
7. [Ablation Study & Training](#ablation) — Paper §II-H
8. [Results & Visualisations](#results) — Paper §III
9. [Extended Metrics](#extended) — active-demand MAE, score distribution
10. [Discussion & Conclusions](#discussion) — Paper §IV, V


<a id='config'></a>
## 1. Configuration

All experiment parameters are defined here. Change paths and hyperparameters in **this single cell** — nothing else needs editing.

> **Running outside Kaggle?** Set `PLATFORM = 'local'` and update `PATH_BEAUTY` / `PATH_ELEC` to your local CSV paths.


In [ ]:
# ── PLATFORM ──────────────────────────────────────────────────────────────────
PLATFORM = 'kaggle'   # 'kaggle' | 'colab' | 'local'

# ── DATA PATHS ────────────────────────────────────────────────────────────────
if PLATFORM == 'kaggle':
    PATH_BEAUTY = '/kaggle/input/multimodal-time-series-forecasting/amazon_review/All_Beauty_f.csv'
    PATH_ELEC   = '/kaggle/input/multimodal-time-series-forecasting/amazon_review/Electronics_f.csv'
elif PLATFORM == 'colab':
    # Mount Drive first: from google.colab import drive; drive.mount('/content/drive')
    PATH_BEAUTY = '/content/drive/MyDrive/data/All_Beauty_f.csv'
    PATH_ELEC   = '/content/drive/MyDrive/data/Electronics_f.csv'
else:  # local
    PATH_BEAUTY = './data/All_Beauty_f.csv'
    PATH_ELEC   = './data/Electronics_f.csv'

# ── DATASET HYPERPARAMETERS ───────────────────────────────────────────────────
N_SEARCH       = 20000   # rows to scan when finding active products
N_PRODUCTS     = 200     # active SKUs per category (Paper §II-C)
N_DAYS         = 100     # trailing days of sales history
DEMAND_THRESH  = 15      # volume threshold — Paper §II-C Eq. 1

# ── MODEL ─────────────────────────────────────────────────────────────────────
SLM_MODEL_ID   = 'microsoft/Phi-3-mini-4k-instruct'
PROMPT_MAX_LEN = 300     # chars of description fed to SLM (above-the-fold heuristic)

# ── RANDOM FOREST ─────────────────────────────────────────────────────────────
RF_N_ESTIMATORS = 100
RANDOM_SEED     = 42     # all stochastic components use this seed

# ── TRAIN / TEST SPLIT ────────────────────────────────────────────────────────
TRAIN_FRACTION  = 0.8    # chronological split ratio

print('Configuration loaded.')
print(f'  Platform      : {PLATFORM}')
print(f'  Products/cat  : {N_PRODUCTS}')
print(f'  Demand thresh : >{DEMAND_THRESH} units / {N_DAYS} days')
print(f'  Random seed   : {RANDOM_SEED}')


<a id='env'></a>
## 2. Environment Setup

`transformers==4.40.2` is pinned for Phi-3-mini compatibility on Kaggle. The critical inference flag `use_cache=False` prevents the `DynamicCache AttributeError` introduced in later versions.

`BitsAndBytes` enables 4-bit NF4 quantization, reducing model VRAM from ~15 GB (FP16) to **2.2 GB**, making the framework deployable on a single NVIDIA Tesla T4.


In [ ]:
# Pinned for Phi-3-mini / DynamicCache compatibility on Kaggle T4
!pip install -q transformers==4.40.2 bitsandbytes accelerate tqdm

import warnings
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings('ignore')

# ── Global reproducibility seed ───────────────────────────────────────────────
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Seed     : {RANDOM_SEED}')


<a id='data'></a>
## 3. Data Acquisition & Preprocessing

**Dataset:** Amazon Product Research Archive 2023  
**Categories:** All Beauty (hedonic/subjective) and Electronics (utilitarian/objective)

Each CSV contains:
- **Metadata columns:** `parent_asin`, `title`, `description`, `average_rating`
- **Time-series columns:** Daily sales counts named as dates (e.g. `2023-01-15`)

### Volume Thresholding Heuristic (Paper Section II-C, Eq. 1)

To eliminate cold-start and dormant products we keep only SKUs where:

$$\sum_{t=1}^{100} d_t > 15$$

This ensures every product has sufficient sales history to form a meaningful baseline. We cap at **200 active products per category** to balance computational speed with statistical significance.


In [ ]:
def load_active_products(path, n_search=N_SEARCH, n_products=N_PRODUCTS,
                         n_days=N_DAYS, demand_thresh=DEMAND_THRESH):
    """
    Load dataset and filter for active products.
    Implements Paper Section II-C, Eq. 1: Volume Thresholding Heuristic (sum > threshold).

    Fix vs draft: uses .nlargest(n_products, 'total_sales') for representative
    sampling by activity rank, not positional .head() which returns CSV-order products.

    Returns: (dataframe with meta + last n_days sales columns, list of date col names)
    """
    print(f'Loading: {path.split("/")[-1]}')
    df_raw = pd.read_csv(path, nrows=n_search)

    # Date columns identified by '-' in column name (e.g. '2023-01-15')
    all_date_cols = [c for c in df_raw.columns if '-' in c]
    date_cols = all_date_cols[-n_days:]   # last n_days of sales history

    meta_cols = ['parent_asin', 'title', 'description', 'average_rating']

    # Volume Thresholding: retain SKUs with total demand > threshold
    df_raw['total_sales'] = df_raw[date_cols].sum(axis=1)
    df_active = (
        df_raw[df_raw['total_sales'] > demand_thresh]
        [meta_cols + date_cols + ['total_sales']]
        .nlargest(n_products, 'total_sales')   # rank by activity, not CSV position
        .drop(columns='total_sales')
        .copy()
    )

    # Sanity check
    assert len(df_active) > 0, f'No active products found in {path}. Lower demand_thresh or increase n_search.'
    print(f'  Active products : {len(df_active)}')
    print(f'  Date range      : {date_cols[0]}  to  {date_cols[-1]}')
    return df_active, date_cols

df_beauty, date_cols_beauty = load_active_products(PATH_BEAUTY)
df_elec,   date_cols_elec   = load_active_products(PATH_ELEC)


<a id='temporal'></a>
## 4. Temporal Alignment: Daily to Weekly Aggregation

**Paper Section II-D, Eq. 2**

Initial experiments at **daily granularity produced 0% improvement** from adding SLM sentiment. The cause: High-Frequency Stochastic Noise — random day-to-day fluctuations from logistics delays, day-of-week effects, and micro-promotions overwhelmed any underlying trend signal.

**Solution:** Aggregate daily demand into weekly buckets anchored to Monday:

$$W_k = \sum_{i=7k-6}^{7k} d_i$$

**Why this works:** Weekly aggregation acts as a **low-pass filter** — it smooths out high-frequency noise and reveals the slow-moving trend that the static SLM sentiment score can meaningfully correlate with. This temporal alignment was the decisive factor behind the **2.83% Beauty improvement**.


In [ ]:
def melt_and_aggregate_weekly(df, date_cols, meta_cols=None):
    """
    Reshape wide-format sales data to long format and aggregate to weekly buckets.
    Implements Paper Section II-D, Eq. 2.
    Weekly anchor: W-MON (Monday) for reproducibility.
    """
    if meta_cols is None:
        meta_cols = ['parent_asin']

    df_long = df.melt(
        id_vars=meta_cols,
        value_vars=date_cols,
        var_name='date',
        value_name='demand'
    )
    df_long['date'] = pd.to_datetime(df_long['date'])

    df_weekly = (
        df_long
        .groupby(['parent_asin', pd.Grouper(key='date', freq='W-MON')])
        .agg({'demand': 'sum'})
        .reset_index()
    )

    assert df_weekly['demand'].isna().sum() == 0, 'NaN demand after aggregation'
    return df_weekly

df_beauty_weekly = melt_and_aggregate_weekly(df_beauty, date_cols_beauty, ['parent_asin', 'description'])
df_elec_weekly   = melt_and_aggregate_weekly(df_elec,   date_cols_elec,   ['parent_asin', 'description'])

print('All Beauty weekly :', df_beauty_weekly.shape)
print('Electronics weekly:', df_elec_weekly.shape)
print(df_beauty_weekly.head())


### 3.1 Visualising the Low-Pass Filter Effect (Paper Fig. 4)

The chart below illustrates the daily-vs-weekly contrast that motivated our temporal design choice.

> **Note:** Uses illustrative synthetic data to demonstrate the noise-smoothing concept, corresponding to Figure 4 in the paper.


In [ ]:
# Illustrative synthetic data — demonstrates the aggregation concept (Paper Fig. 4)
np.random.seed(42)
days = np.arange(1, 101)
daily_sales  = 5 + 0.05 * days + np.random.poisson(3, 100)
weekly_sales = [np.sum(daily_sales[i:i+7]) for i in range(0, len(daily_sales), 7)]
weekly_days  = np.arange(1, len(weekly_sales) + 1) * 7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Temporal Aggregation as a Low-Pass Filter — Paper Fig. 4 (Illustrative)', fontsize=13, y=1.02)

axes[0].bar(days, daily_sales, color='lightgray', alpha=0.7)
axes[0].plot(days, daily_sales, color='red', linewidth=1, alpha=0.6)
axes[0].set_title('Daily Granularity: High Stochastic Noise\n(Result: 0% Improvement)', fontsize=12)
axes[0].set_xlabel('Days'); axes[0].set_ylabel('Units Sold')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].step(weekly_days, weekly_sales, where='mid', color='steelblue', linewidth=2.5)
axes[1].fill_between(weekly_days, weekly_sales, step='mid', alpha=0.15, color='steelblue')
axes[1].set_title('Weekly Granularity: Signal Smoothing\n(Result: 2.83% Improvement)', fontsize=12)
axes[1].set_xlabel('Weeks (Aggregated)'); axes[1].set_ylabel('Total Weekly Units Sold')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('fig4_temporal_aggregation.png', dpi=300, bbox_inches='tight')
plt.show()


<a id='slm'></a>
## 5. SLM Semantic Inference — Phi-3-mini (4-bit NF4)

**Paper Section II-E**

### 5.1 Why Phi-3-mini? (Efficiency-Accuracy Trade-off)

Most prior work deploys large, cloud-dependent LLMs. We chose **Microsoft Phi-3-mini-4k-instruct (3.8B parameters)** because:
- Reasoning capabilities comparable to models 10x its size
- Recognises domain-specific subtleties (e.g. 'clinical-grade hyaluronic acid' signals higher demand elasticity than 'moisturising cream')
- Runs on commodity hardware after quantisation

### 5.2 4-bit NF4 Quantisation (Paper Section II-E-1)

| Configuration | Value |
|---|---|
| Quantisation type | NormalFloat4 (NF4) |
| Compute dtype | torch.float16 (hybrid precision) |
| Model size FP16 | ~15 GB |
| Model size after NF4 | ~2.2 GB |
| GPU | NVIDIA Tesla T4 (16 GB VRAM) |

Hybrid precision: weights stored in 4-bit, computation in float16 — preserving enough numerical precision to distinguish a 9/10 appeal score from a 7/10.

### 5.3 Zero-Shot Heuristic Prompting (Paper Section II-E-2)

The SLM acts as a **professional retail analyst**, scoring each product's **Market Appeal Index S ∈ {1,...,10}** from its description using zero-shot inference. This is a latent demand signal invisible to purely historical models.


In [ ]:
print('Loading Phi-3-mini with 4-bit NF4 quantisation...')
model_id = 'microsoft/Phi-3-mini-4k-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',             # NormalFloat4 (Paper Section II-E-1)
    bnb_4bit_compute_dtype=torch.float16,  # Hybrid precision
    bnb_4bit_use_double_quant=False
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model_phi3 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map='auto',
    attn_implementation='eager'  # Stable attention on T4
)

if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated() / 1e9
    print(f'Model loaded. VRAM used: {mem_gb:.2f} GB  (paper target: ~2.2 GB)')


### 4.4 Market Appeal Index Extraction Function

**Prompt design:** Retail analyst persona + structured integer output constraint.  
**Fallback:** Score of **5** (neutral) when description is absent or too short — avoids injecting spurious signal.  
**Key fix:** `use_cache=False` resolves the `DynamicCache AttributeError` in transformers 4.40.x on Kaggle.


In [ ]:
def get_market_appeal_score(text, tokenizer, model):
    """
    Extract Market Appeal Index S (1-10) from product description.
    Implements Paper Section II-E-2: Zero-Shot Heuristic Prompting.

    Design choices:
    - Neutral fallback of 5 (not cyclic index) to avoid spurious signal
    - use_cache=False: fixes DynamicCache AttributeError on transformers 4.40.x
    - max_new_tokens=1: enforces single-digit output, minimises latency
    - do_sample=False: deterministic scoring for reproducibility
    - try/except: recovers gracefully from CUDA/OOM errors on individual products
    """
    if not isinstance(text, str) or len(text.strip()) < 10:
        return 5  # Neutral fallback — short/missing description

    prompt = (
        '<|user|>You are a professional retail analyst. '
        'Based on the following product description, assign a Market Appeal Index (1-10) '
        'reflecting commercial desirability, premium positioning, and consumer emotional appeal. '
        'Return ONLY the integer.\n\nProduct: ' + text[:PROMPT_MAX_LEN] + '<|end|><|assistant|>'
    )

    try:
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1,
                do_sample=False,   # Deterministic
                use_cache=False    # Critical fix for DynamicCache AttributeError
            )
        response = tokenizer.decode(outputs[0][-1:], skip_special_tokens=True).strip()
        digits = ''.join(filter(str.isdigit, response))
        if digits:
            return max(1, min(10, int(digits[0])))  # Clamp to valid range
        return 5  # Non-digit response fallback

    except Exception as e:
        print(f'  [WARN] SLM inference failed ({type(e).__name__}): returning neutral score 5')
        return 5

print('Market Appeal Index extractor ready.')
print('Fallback: 5 (neutral) when description is unavailable, too short, or inference fails.')


### 4.5 Run Inference Across Both Categories

One score per unique `parent_asin`. The sentiment score is a **static product-level feature** — the product description does not change week to week. It is broadcast across all weekly time steps via a left-join in the fusion step.


In [ ]:
def extract_sentiment_scores(df_weekly, tokenizer, model, category_name):
    """
    Extract Market Appeal Index for every unique product in a category.
    Static per product: score is broadcast across all weekly rows in the fusion step.
    Uses the `model` parameter (not a global) for clean encapsulation.
    """
    print(f'\nExtracting Market Appeal Index — {category_name}')
    unique_prods = df_weekly[['parent_asin', 'description']].drop_duplicates().copy()
    print(f'  Unique products: {len(unique_prods)}')

    scores = []
    for _, row in tqdm(unique_prods.iterrows(), total=len(unique_prods), desc=category_name):
        scores.append(get_market_appeal_score(row['description'], tokenizer, model))

    unique_prods['sentiment_score'] = scores

    # Sanity check: every product must have a score
    assert unique_prods['sentiment_score'].isna().sum() == 0, 'NaN sentiment scores detected'
    assert len(unique_prods) == df_weekly['parent_asin'].nunique(), \
        'Score count mismatch vs unique products in weekly data'

    print(f'  Score distribution:\n{unique_prods["sentiment_score"].value_counts().sort_index().to_string()}')
    return unique_prods[['parent_asin', 'sentiment_score']]

scores_beauty = extract_sentiment_scores(df_beauty_weekly, tokenizer, model_phi3, 'All Beauty')
scores_elec   = extract_sentiment_scores(df_elec_weekly,   tokenizer, model_phi3, 'Electronics')


<a id='fusion'></a>
## 6. Multimodal Feature Fusion — Late Fusion Architecture

**Paper Section II-F & II-G**

The framework fuses two complementary signals:

| Feature | Type | Update Frequency | Description |
|---|---|---|---|
| `lag_1_week` | Dynamic temporal | Weekly | Previous week's demand volume |
| `sentiment_score` | Static semantic | Once per product | Phi-3-mini Market Appeal Index |

**Late Fusion:** The two signals remain separate until the Random Forest learns their interactions. The RF discovers non-linear rules such as: *'when lag-1 sales are rising AND sentiment score >= 8, forecast a significant demand spike.'*

**Train/Test Split:** Chronological split by **date threshold** (not row index) to prevent cross-product data leakage — different products' rows can interleave under a row-based split.


In [ ]:
def build_feature_matrix(df_weekly, sentiment_scores, train_fraction=TRAIN_FRACTION):
    """
    Build multimodal feature matrix: lag_1_week + sentiment_score.
    Implements Paper Section II-F: Late Fusion Architecture.
    Chronological date-based split prevents cross-product data leakage.
    """
    # Broadcast static sentiment score across all weekly rows (left join)
    df = df_weekly.merge(sentiment_scores, on='parent_asin', how='left')
    df['sentiment_score'] = df['sentiment_score'].fillna(5).astype(float)

    # Lag-1 feature: previous week's demand (Paper Section II-F)
    df = df.sort_values(['parent_asin', 'date'])
    df['lag_1_week'] = df.groupby('parent_asin')['demand'].shift(1)
    df = df.dropna(subset=['lag_1_week']).reset_index(drop=True)

    # Chronological 80/20 split by date — avoids cross-product leakage
    cutoff = df['date'].quantile(train_fraction)
    train = df[df['date'] <= cutoff].copy()
    test  = df[df['date'] >  cutoff].copy()

    # Pipeline integrity checks
    assert len(train) > 0 and len(test) > 0, 'Train or test set is empty after split'
    assert not train['lag_1_week'].isna().any(), 'NaN lag values in train set'
    assert not test['lag_1_week'].isna().any(),  'NaN lag values in test set'

    print(f'  Total rows   : {len(df)}')
    print(f'  Train rows   : {len(train)}  (up to {cutoff.date()})')
    print(f'  Test rows    : {len(test)}')
    return train, test

print('All Beauty')
train_beauty, test_beauty = build_feature_matrix(df_beauty_weekly, scores_beauty)
print('\nElectronics')
train_elec, test_elec = build_feature_matrix(df_elec_weekly, scores_elec)


<a id='ablation'></a>
## 7. Ablation Study & Model Training

**Paper Section II-G-1 & II-H-2**

### Why Random Forest?
1. **Non-linear interaction learning** — captures 'rising sales AND high sentiment => spike' rules
2. **Heteroscedasticity robustness** — bagging smooths retail data's variable variance
3. **Gini Importance interpretability** — quantifies how much the SLM score actually contributed
4. **Appropriate for 200-product sample** — avoids LSTM/Transformer overfitting on small datasets

### Ablation Design

| Model | Features | Purpose |
|---|---|---|
| Baseline | `lag_1_week` only | Historical sales signal alone |
| Multimodal | `lag_1_week` + `sentiment_score` | History + SLM semantic signal |

**Metric:** Mean Absolute Error (Paper Eq. 3) — interpretable as missed sales units per week.


In [ ]:
def run_ablation(train, test, category_name,
                 n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_SEED):
    """
    Ablation study: Baseline vs Multimodal.
    Implements Paper Section II-H-2 and Eq. 3 (MAE).
    """
    print(f'\n{"="*52}')
    print(f'  Ablation Study -- {category_name}')
    print(f'{"="*52}')

    y_train, y_test = train['demand'], test['demand']

    # Experiment A: Baseline — history only
    rf_base = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    rf_base.fit(train[['lag_1_week']], y_train)
    preds_base = rf_base.predict(test[['lag_1_week']])
    mae_base   = mean_absolute_error(y_test, preds_base)

    # Experiment B: Multimodal — history + SLM Market Appeal Index
    rf_multi = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    rf_multi.fit(train[['lag_1_week', 'sentiment_score']], y_train)
    preds_multi = rf_multi.predict(test[['lag_1_week', 'sentiment_score']])
    mae_multi   = mean_absolute_error(y_test, preds_multi)

    improvement = ((mae_base - mae_multi) / mae_base) * 100
    direction   = 'IMPROVEMENT ✅' if improvement > 0 else 'ERROR INCREASE ❌'

    print(f'  Baseline MAE   (Sales Only)     : {mae_base:.4f}')
    print(f'  Multimodal MAE (Sales + Phi-3)  : {mae_multi:.4f}')
    print(f'  MAE Change                      : {improvement:+.2f}%  [{direction}]')

    # Feature importance — how much did the SLM score contribute?
    fi = rf_multi.feature_importances_
    print(f'\n  Feature Importances (Multimodal RF):')
    print(f'    lag_1_week      : {fi[0]:.4f}  ({fi[0]*100:.1f}%)')
    print(f'    sentiment_score : {fi[1]:.4f}  ({fi[1]*100:.1f}%)')

    return {
        'mae_base': mae_base, 'mae_multi': mae_multi, 'improvement': improvement,
        'preds_base': preds_base, 'preds_multi': preds_multi,
        'rf_base': rf_base, 'rf_multi': rf_multi,
        'y_test': y_test, 'test': test
    }

results_beauty = run_ablation(train_beauty, test_beauty, 'All Beauty')
results_elec   = run_ablation(train_elec,   test_elec,   'Electronics')


<a id='results'></a>
## 8. Results & Visualisations

**Paper Section III**

### Summary of Findings

| Category | Baseline MAE | Multimodal MAE | Change | Interpretation |
|---|---|---|---|---|
| All Beauty | 4.59 | 4.46 | **−2.83% ✅** | Sentiment = strong demand signal |
| Electronics | 3.20 | 3.31 | **+3.44% ❌** | Sentiment = semantic noise |

**All Beauty:** Beauty products are **hedonic goods** — demand is emotionally driven. Terms like 'organic', 'anti-aging', and 'glowing' are predictive of sustained purchase intent. The SLM captures this latent desire signal.

**Electronics:** Electronics are **utilitarian goods** — consumers decide on objective specs ('128GB storage', '5000mAh battery'). The abstract appeal score adds noise, slightly worsening the forecast.

> **Central contribution:** Sentiment-based enrichment is domain-sensitive. Apply multimodal AI selectively to hedonic product categories.


In [ ]:
# Paper Fig. 2 & 3 -- Ablation results and forecast comparison
categories   = ['All Beauty', 'Electronics']
mae_base_all = [results_beauty['mae_base'], results_elec['mae_base']]
mae_mult_all = [results_beauty['mae_multi'], results_elec['mae_multi']]
improvements = [results_beauty['improvement'], results_elec['improvement']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Ablation Study: Impact of SLM Sentiment on Forecast Error', fontsize=14)

# Left: MAE comparison bar chart
x, width = np.arange(len(categories)), 0.35
axes[0].bar(x - width/2, mae_base_all, width, label='Baseline (Sales Only)',    color='#A0A0A0', alpha=0.85)
axes[0].bar(x + width/2, mae_mult_all, width, label='Multimodal (Sales + SLM)', color='#2E86C1', alpha=0.85)
for i, (imp, xpos) in enumerate(zip(improvements, x)):
    colour = 'green' if imp > 0 else 'red'
    axes[0].text(xpos, max(mae_base_all[i], mae_mult_all[i]) + 0.08,
                 f'{imp:+.2f}%', ha='center', color=colour, fontweight='bold', fontsize=12)
axes[0].set_xlabel('Product Category')
axes[0].set_ylabel('Mean Absolute Error (lower is better)')
axes[0].set_title('MAE: Baseline vs Multimodal (Paper Fig. 2 & 3)')
axes[0].set_xticks(x); axes[0].set_xticklabels(categories)
axes[0].legend(); axes[0].set_ylim(0, max(mae_base_all) * 1.4)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Right: Forecast vs actual -- Beauty category (first 50 test points)
actual = results_beauty['y_test'].values[:50]
p_base = results_beauty['preds_base'][:50]
p_mult = results_beauty['preds_multi'][:50]
axes[1].plot(actual, label='Actual Demand',  color='black',   linewidth=2, marker='o', markersize=3)
axes[1].plot(p_base, label=f'Baseline  (MAE: {results_beauty["mae_base"]:.4f})',
             color='gray', linewidth=1.5, linestyle='--', alpha=0.8)
axes[1].plot(p_mult, label=f'Multimodal (MAE: {results_beauty["mae_multi"]:.4f})',
             color='#2E86C1', linewidth=2)
axes[1].text(0.02, 0.95, f'Error Reduction: {results_beauty["improvement"]:+.2f}%',
             transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
axes[1].set_title('Forecast Accuracy: All Beauty -- Weekly')
axes[1].set_xlabel('Test Samples (Weekly Time Steps)')
axes[1].set_ylabel('Demand Volume')
axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig2_ablation_results.png', dpi=300, bbox_inches='tight')
plt.show()


### 8.1 Domain Sensitivity Analysis (Paper Fig. 5)

Semantic correlation between SLM score and demand confirms the hedonic/utilitarian distinction.


In [ ]:
# Domain sensitivity: correlation and stratified demand by sentiment score
print('Domain Sensitivity Summary')
print('=' * 56)
print(f'{"Metric":<35} {"Beauty":>8} {"Electronics":>12}')
print('-' * 56)
print(f'{"Baseline MAE":<35} {results_beauty["mae_base"]:>8.4f} {results_elec["mae_base"]:>12.4f}')
print(f'{"Multimodal MAE":<35} {results_beauty["mae_multi"]:>8.4f} {results_elec["mae_multi"]:>12.4f}')
print(f'{"MAE Change (%)":<35} {results_beauty["improvement"]:>+8.2f} {results_elec["improvement"]:>+12.2f}')
print('=' * 56)

for label, train_df in [('All Beauty', train_beauty), ('Electronics', train_elec)]:
    corr = train_df[['sentiment_score', 'demand']].corr().iloc[0, 1]
    print(f'\nSentiment-Demand Correlation ({label}): {corr:.4f}')
    print(f'Mean weekly demand by SLM score:\n{train_df.groupby("sentiment_score")["demand"].mean().to_string()}')


<a id='extended'></a>
## 9. Extended Metrics

Beyond the headline MAE, two supplementary analyses validate the paper's core claims:

**9.1 Active-Demand MAE** — re-evaluates both models on non-zero demand weeks only,  
removing weeks where a product was simply out of stock or inactive. This gives a cleaner  
picture of accuracy when demand actually exists.

**9.2 Score Distribution Heatmap** — visualises how the SLM's Market Appeal Index  
distributes across both categories, confirming the scoring function doesn't degenerate  
to a constant or narrow band.


In [ ]:
from sklearn.metrics import mean_absolute_error

# ── 9.1 Active-Demand MAE ──────────────────────────────────────────────────────
print('9.1  Active-Demand MAE (non-zero demand weeks only)')
print('='*56)

for label, results, test_df in [
    ('All Beauty',   results_beauty, test_beauty),
    ('Electronics',  results_elec,   test_elec),
]:
    active = test_df[test_df['demand'] > 0].copy()
    if active.empty:
        print(f'  {label}: no active weeks in test set'); continue

    mae_b_act = mean_absolute_error(
        active['demand'],
        results['rf_base'].predict(active[['lag_1_week']])
    )
    mae_m_act = mean_absolute_error(
        active['demand'],
        results['rf_multi'].predict(active[['lag_1_week', 'sentiment_score']])
    )
    lift = ((mae_b_act - mae_m_act) / mae_b_act) * 100
    direction = '✅' if lift > 0 else '❌'

    print(f'  {label}')
    print(f'    Active weeks          : {len(active)} / {len(test_df)} ({100*len(active)/len(test_df):.0f}%)')
    print(f'    Baseline MAE (active) : {mae_b_act:.4f}')
    print(f'    Multimodal MAE (active): {mae_m_act:.4f}')
    print(f'    Lift (active only)    : {lift:+.2f}% {direction}')
    print()

# ── 9.2 Score Distribution Heatmap ────────────────────────────────────────────
print('9.2  SLM Market Appeal Index — Score Distribution')
print('='*56)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Market Appeal Index Distribution by Category', fontsize=13)

for ax, (label, train_df) in zip(axes, [('All Beauty', train_beauty), ('Electronics', train_elec)]):
    counts = train_df.groupby('sentiment_score')['demand'].agg(['mean', 'count']).reset_index()
    counts.columns = ['score', 'mean_demand', 'n_weeks']

    bars = ax.bar(counts['score'], counts['n_weeks'], color='#2E86C1', alpha=0.7, edgecolor='white')
    ax2  = ax.twinx()
    ax2.plot(counts['score'], counts['mean_demand'], color='#E74C3C', marker='o',
             linewidth=2, label='Mean weekly demand')
    ax2.set_ylabel('Mean Weekly Demand', color='#E74C3C', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#E74C3C')

    ax.set_title(f'{label}', fontsize=12)
    ax.set_xlabel('Market Appeal Score (1–10)')
    ax.set_ylabel('Number of Product-Weeks', color='#2E86C1')
    ax.tick_params(axis='y', labelcolor='#2E86C1')
    ax.set_xticks(range(1, 11))
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    corr = train_df[['sentiment_score', 'demand']].corr().iloc[0, 1]
    ax.text(0.03, 0.95, f'Pearson r = {corr:.4f}', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('fig_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


<a id='discussion'></a>
## 10. Discussion & Conclusions

**Paper Section IV & V**

### Three Key Lessons for Supply Chain Practitioners

**1. Sentiment is domain-specific.**  
SLM-derived sentiment is a useful demand-sensing signal for subjective goods (Beauty, Fashion) but acts as noise for objective goods (Electronics, Hardware). Apply multimodal enrichment selectively.

**2. Temporal aggregation is non-negotiable.**  
The jump from 0% (daily) to 2.83% (weekly) shows that semantic features need smooth, noise-reduced data to contribute. Daily stochastic volatility drowns the sentiment signal entirely.

**3. SLMs are enterprise supply chain-ready.**  
Phi-3-mini in 4-bit NF4 on a single Kaggle T4 GPU (2.2 GB VRAM footprint) proves that sentiment-aware demand forecasting is achievable without costly GPU infrastructure.

### Limitations & Future Work
- **Static sentiment:** Currently scored once from product descriptions. Future work should incorporate dynamic sentiment from rolling customer reviews.
- **Category expansion:** Mixed categories (Home Decor, Apparel) need further study to determine the sentiment signal/noise boundary.
- **Early fusion:** Simultaneously processing text and numerical features in a neural architecture may outperform this late-fusion Random Forest approach.

---

### Citation

If you use this work, please cite:

```bibtex
@article{tiple2025predictive,
  title={A Predictive Supply Chain Framework: Leveraging Small Language Models
         for Sentiment-Based Demand Forecasting},
  author={Tiple, Bhavana and Kulkarni, Riya and Panchal, Naitri and Churi, Heramb},
  institution={MIT World Peace University},
  year={2025}
}
```
